In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
# Configuration des graphiques
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_rows', None)
#models
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report ,recall_score, precision_score


In [ ]:
df = pd.read_csv('data_clean.csv',low_memory=False)

In [ ]:
cols_to_drop = ['purpose', 'purpose_step1', 'purpose_risk_class', 'earliest_cr_line_dt']
df = df.drop(columns=cols_to_drop)

In [ ]:
# 1. Nettoyer le dataset principal
missing_cols = ['open_act_il', 'open_il_24m', 'total_bal_il', 'open_rv_12m',
                'open_rv_24m', 'inq_fi', 'total_cu_tl', 'inq_last_12m']

# Variables de comptage → 0
count_vars = ['open_act_il', 'open_il_24m', 'open_rv_12m', 'open_rv_24m',
              'inq_fi', 'total_cu_tl', 'inq_last_12m']
df[count_vars] = df[count_vars].fillna(0)

# Variable continue → médiane
df['total_bal_il'] = df['total_bal_il'].fillna(df['total_bal_il'].median())

# Dernier Analyse

In [ ]:
df = df.dropna(subset=['default_risk'])

In [ ]:
# Gestion complète des valeurs manquantes
import pandas as pd

# 1. Variables numériques
count_vars = ['open_acc_6m', 'open_il_12m']  # Comptage → 0
cont_vars = ['dti', 'all_util', 'max_bal_bc']  # Continue → médiane

# Créer indicatrices pour variables importantes
for col in cont_vars:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

# Imputation
df[count_vars] = df[count_vars].fillna(0)
df[cont_vars] = df[cont_vars].fillna(df[cont_vars].median())

# 2. Variable catégorielle
df['home_ownership_ordinal'] = df['home_ownership_ordinal'].fillna(1.0)

print(f"Valeurs manquantes finales: {df.isnull().sum().sum()}")

Valeurs manquantes finales: 0


# Modeling

In [ ]:
X = df.drop('default_risk', axis=1)
y = df['default_risk']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
X_train = X_train.drop(columns=categorical_cols)
X_test = X_test.drop(columns=categorical_cols)

In [ ]:

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")
print(f"Distribution train: {y_train.value_counts()}")
print(f"Distribution test: {y_test.value_counts()}")

Train: (9719, 103)
Test: (2430, 103)
Distribution train: default_risk
0.0    7820
1.0    1899
Name: count, dtype: int64
Distribution test: default_risk
0.0    1955
1.0     475
Name: count, dtype: int64


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,    # 100 arbres
    random_state=42
)

rf_model.fit(X_train, y_train)

# Prédire
y_pred = rf_model.predict(X_test)

# Résultats simples
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Précision: {accuracy:.3f}")

print("\n📊 Rapport détaillé:")
print(classification_report(y_test, y_pred))

✅ Précision: 0.837

📊 Rapport détaillé:
              precision    recall  f1-score   support

         0.0       0.84      0.99      0.91      1955
         1.0       0.84      0.21      0.33       475

    accuracy                           0.84      2430
   macro avg       0.84      0.60      0.62      2430
weighted avg       0.84      0.84      0.80      2430



Problème avec ce résultat : le modèle repère bien les bons clients, mais rate environ 80% des mauvais payeurs. Prochaine étape : donner plus de poids au défaut via le seuil de décision et les réglages du Random Forest.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


y_proba = rf_model.predict_proba(X_test)[:, 1]
seuil = 0.3  #
y_pred = (y_proba >= seuil).astype(int)

# Le reste reste identique
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Précision: {accuracy:.3f}")
print("\n📊 Rapport détaillé:")
print(classification_report(y_test, y_pred))

✅ Précision: 0.781

📊 Rapport détaillé:
              precision    recall  f1-score   support

         0.0       0.88      0.85      0.86      1955
         1.0       0.45      0.50      0.47       475

    accuracy                           0.78      2430
   macro avg       0.66      0.68      0.67      2430
weighted avg       0.79      0.78      0.79      2430



In [ ]:
from sklearn.metrics import recall_score, precision_score
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# NOUVEAU: Tester plusieurs seuils
y_proba = rf_model.predict_proba(X_test)[:, 1]

seuils_a_tester = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

for seuil in seuils_a_tester:
    y_pred = (y_proba >= seuil).astype(int)

    # Calculer seulement rappel et précision classe 1
    recall_1 = recall_score(y_test, y_pred, pos_label=1)
    precision_1 = precision_score(y_test, y_pred, pos_label=1)

    print(f"Seuil {seuil}: Rappel={recall_1:.2f} | Précision={precision_1:.2f}")

Seuil 0.2: Rappel=0.73 | Précision=0.33
Seuil 0.25: Rappel=0.60 | Précision=0.38
Seuil 0.3: Rappel=0.50 | Précision=0.45
Seuil 0.35: Rappel=0.40 | Précision=0.53
Seuil 0.4: Rappel=0.33 | Précision=0.63
Seuil 0.45: Rappel=0.25 | Précision=0.73
Seuil 0.5: Rappel=0.21 | Précision=0.83


Les seuils 0.25 et 0.3 semblent les plus raisonnables, mais ils déclenchent aussi des actions de recouvrement inutiles sur des clients qui auraient remboursé normalement.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced' )
rf_model.fit(X_train, y_train)


y_proba = rf_model.predict_proba(X_test)[:, 1]
seuil = 0.2  #
y_pred = (y_proba >= seuil).astype(int)

# Le reste reste identique
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Précision: {accuracy:.3f}")
print("\n📊 Rapport détaillé:")
print(classification_report(y_test, y_pred))

✅ Précision: 0.705

📊 Rapport détaillé:
              precision    recall  f1-score   support

         0.0       0.91      0.71      0.79      1955
         1.0       0.37      0.70      0.48       475

    accuracy                           0.71      2430
   macro avg       0.64      0.70      0.64      2430
weighted avg       0.80      0.71      0.73      2430



Sur 100 clients à risque, on en détecte maintenant 71, contre 21 avec la configuration précédente.

In [ ]:
seuil = 0.2
resultats = []

# Boucle de test sur différents nombres d'arbres
for n_trees in [100, 200, 300, 500]:
    print(f"Testing {n_trees} arbres...", end=" ")

    # Modèle
    rf = RandomForestClassifier(
        n_estimators=n_trees,
        class_weight='balanced',
        random_state=42
    )

    # Entraînement
    rf.fit(X_train, y_train)

    # Prédictions
    y_proba = rf.predict_proba(X_test)[:, 1]  # Probabilité pour la classe positive
    y_pred = (y_proba >= seuil).astype(int)   # Seuil de décision

    # Métriques
    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)

    # Sauvegarde des résultats
    resultats.append({
        'Paramètre': f'n_estimators={n_trees}',
        'Rappel': recall,
        'Précision': precision,
        'Type': 'n_estimators'
    })

    print(f"Rappel: {recall:.3f} | Précision: {precision:.3f}")

Testing 100 arbres... Rappel: 0.697 | Précision: 0.367
Testing 200 arbres... Rappel: 0.697 | Précision: 0.367
Testing 300 arbres... Rappel: 0.676 | Précision: 0.363
Testing 500 arbres... Rappel: 0.680 | Précision: 0.364


In [ ]:
best_trees = 500

for depth in [10, 15, 20, None]:
    print(f"Testing profondeur {depth}...", end=" ")

    rf = RandomForestClassifier(
       n_estimators=best_trees,
       max_depth=depth,
       class_weight='balanced',
       random_state=42
        )

    rf.fit(X_train, y_train)
    y_proba = rf.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= seuil).astype(int)

    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)

    resultats.append({
            'Paramètre': f'max_depth={depth}',
            'Rappel': recall,
            'Précision': precision,
            'Type': 'max_depth'
        })

    print(f"Rappel: {recall:.3f} | Précision: {precision:.3f}")

Testing profondeur 10... Rappel: 0.960 | Précision: 0.232
Testing profondeur 15... Rappel: 0.825 | Précision: 0.304
Testing profondeur 20... Rappel: 0.703 | Précision: 0.348
Testing profondeur None... Rappel: 0.680 | Précision: 0.364


Même problème ici : beaucoup de mauvais payeurs restent confondus avec les bons clients.

In [ ]:
for min_split in [2, 5, 10, 20]:
        print(f"Testing min_samples_split {min_split}...", end=" ")

        rf = RandomForestClassifier(
            n_estimators=best_trees,
            max_depth=12,
            min_samples_split=min_split,
            class_weight='balanced',
            random_state=42
        )

        rf.fit(X_train, y_train)
        y_proba = rf.predict_proba(X_test)[:, 1]
        y_pred = (y_proba >= seuil).astype(int)

        recall = recall_score(y_test, y_pred, pos_label=1)
        precision = precision_score(y_test, y_pred, pos_label=1)

        resultats.append({
            'Paramètre': f'min_samples_split={min_split}',
            'Rappel': recall,
            'Précision': precision,
            'Type': 'min_samples_split'
        })

        print(f"Rappel: {recall:.3f} | Précision: {precision:.3f}")


Testing min_samples_split 2... Rappel: 0.920 | Précision: 0.258
Testing min_samples_split 5... Rappel: 0.933 | Précision: 0.254
Testing min_samples_split 10... Rappel: 0.945 | Précision: 0.248
Testing min_samples_split 20... Rappel: 0.964 | Précision: 0.241


Test avec XGBoost

In [ ]:
from xgboost import XGBClassifier


xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss'  # évite les warnings
)

xgb_model.fit(X_train, y_train)
y_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.2).astype(int)  # même seuil que RF

recall = recall_score(y_test, y_pred, pos_label=1)
precision = precision_score(y_test, y_pred, pos_label=1)
print(f"XGBoost de base: Rappel={recall:.3f} | Précision={precision:.3f}")

XGBoost de base: Rappel=0.531 | Précision=0.394


In [ ]:

scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"Scale pos weight: {scale_pos_weight:.2f}")

xgb_balanced = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_balanced.fit(X_train, y_train)
y_proba = xgb_balanced.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.2).astype(int)

recall = recall_score(y_test, y_pred, pos_label=1)
precision = precision_score(y_test, y_pred, pos_label=1)
print(f"XGBoost pondéré: Rappel={recall:.3f} | Précision={precision:.3f}")

Scale pos weight: 4.12
XGBoost pondéré: Rappel=0.697 | Précision=0.319


In [ ]:
scale_pos_weight = 4.17

xgb_balanced = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_balanced.fit(X_train, y_train)
y_proba = xgb_balanced.predict_proba(X_test)[:, 1]

for seuil in [0.15, 0.18, 0.20, 0.25, 0.30]:
    y_pred = (y_proba >= seuil).astype(int)
    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)
    print(f"Seuil {seuil}: Rappel={recall:.3f} | Précision={precision:.3f}")

Seuil 0.15: Rappel=0.754 | Précision=0.304
Seuil 0.18: Rappel=0.716 | Précision=0.315
Seuil 0.2: Rappel=0.684 | Précision=0.320
Seuil 0.25: Rappel=0.627 | Précision=0.337
Seuil 0.3: Rappel=0.581 | Précision=0.358


In [ ]:
print(" Test n_estimators avec seuil 0.15:")

for n_trees in [100, 200, 300, 500]:
    xgb_model = XGBClassifier(
        n_estimators=n_trees,
        scale_pos_weight=4.17,
        random_state=42,
        eval_metric='logloss'
    )

    xgb_model.fit(X_train, y_train)
    y_proba = xgb_model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.15).astype(int)

    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)
    print(f"{n_trees} arbres: Rappel={recall:.3f} | Précision={precision:.3f}")

 Test n_estimators avec seuil 0.15:
100 arbres: Rappel=0.754 | Précision=0.304
200 arbres: Rappel=0.583 | Précision=0.339
300 arbres: Rappel=0.524 | Précision=0.371
500 arbres: Rappel=0.476 | Précision=0.408


Il faut ajouter de nouvelles colonnes et features.

In [ ]:

from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Modèle simple
rf = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# Feature importance
importances = pd.DataFrame({
    'variable': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Top 10
print("Variables les plus importantes:")
print(importances.head(10))



Variables les plus importantes:
                        variable  importance
3                       int_rate    0.068127
99  debt_settlement_flag_encoded    0.047805
6                            dti    0.028003
90                  term_encoded    0.024056
40                bc_open_to_buy    0.021402
4                    installment    0.020523
39                   avg_cur_bal    0.019283
71               tot_hi_cred_lim    0.019078
45          mo_sin_old_rev_tl_op    0.019041
32                    max_bal_bc    0.018178


In [ ]:
# Regarde la distribution complète :
print(f"Top 10: {importances.head(10)['importance'].sum():.1%}")
print(f"Top 20: {importances.head(20)['importance'].sum():.1%}")
print(f"Variables >1%: {(importances['importance'] > 0.01).sum()}")

Top 10: 28.5%
Top 20: 46.1%
Variables >1%: 41


Ratios financiers puissants

In [ ]:
# Option 1: Remplacer par médiane quand annual_inc = 0
df['payment_burden'] = np.where(
    df['annual_inc'] > 0,
    df['installment'] / df['annual_inc'],
    df['installment'] / df['annual_inc'].median()
)

df['debt_stress_score'] = np.where(
    df['annual_inc'] > 0,
    df['int_rate'] * df['dti'] / df['annual_inc'],
    0  # ou une autre valeur par défaut
)

Features d'interaction critiques

Features temporelles

In [ ]:
# Basé sur credit_history_years qui est important
df['experience_vs_debt'] = df['credit_history_years'] / (df['dti'] + 1)  # Maturité crédit
df['recent_vs_total_inquiries'] = df['inq_last_6mths'] / (df['total_acc'] + 1)

Agrégations intelligentes

In [ ]:
# Moyennes de variables importantes
df['total_debt_burden'] = df['tot_cur_bal'] + df['installment'] * df['term_encoded']
df['financial_health_score'] = df['annual_inc'] / (df['tot_cur_bal'] + df['installment'])

In [ ]:
# Refaire le split avec le dataset enrichi
X = df.drop(['default_risk'], axis=1)  # Toutes les features (anciennes + nouvelles)
y = df['default_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Nouvelles dimensions: X_train {X_train.shape}, X_test {X_test.shape}")

Nouvelles dimensions: X_train (9719, 109), X_test (2430, 109)


In [ ]:
# Supprimer les colonnes catégorielles originales (déjà encodées ailleurs)
columns_to_remove = ['purpose', 'purpose_step1', 'purpose_risk_class', 'earliest_cr_line_dt']

print("🗑️ Suppression des colonnes dupliquées:")
for col in columns_to_remove:
    if col in X_train.columns:
        X_train = X_train.drop(col, axis=1)
        X_test = X_test.drop(col, axis=1)
        print(f"    ✅ {col} supprimée")
    else:
        print(f"    ⚠️ {col} déjà absente")

# Vérification finale
print(f"\n📊 Dimensions finales: X_train {X_train.shape}, X_test {X_test.shape}")
print(f"✅ Colonnes texte restantes: {X_train.select_dtypes(include=['object']).columns.tolist()}")

🗑️ Suppression des colonnes dupliquées:
    ⚠️ purpose déjà absente
    ⚠️ purpose_step1 déjà absente
    ⚠️ purpose_risk_class déjà absente
    ⚠️ earliest_cr_line_dt déjà absente

📊 Dimensions finales: X_train (9719, 109), X_test (2430, 109)
✅ Colonnes texte restantes: []


In [ ]:
# Trouve les coupables !
import numpy as np

# Vérifie chaque nouvelle feature
new_features = ['payment_burden', 'credit_utilization_ratio', 'available_credit_ratio',
                'debt_stress_score', 'experience_vs_debt', 'recent_vs_total_inquiries',
                'total_debt_burden', 'financial_health_score']

for feat in new_features:
    if feat in X_train.columns:
        inf_count = np.isinf(X_train[feat]).sum()
        nan_count = X_train[feat].isnull().sum()
        max_val = X_train[feat].max()
        print(f"{feat}: {inf_count} inf, {nan_count} NaN, max={max_val}")

payment_burden: 0 inf, 0 NaN, max=0.13352542372881357
debt_stress_score: 0 inf, 0 NaN, max=10.080305084745762
experience_vs_debt: 0 inf, 0 NaN, max=22.24777549623545
recent_vs_total_inquiries: 0 inf, 0 NaN, max=0.6
total_debt_burden: 0 inf, 0 NaN, max=2170272.88
financial_health_score: 0 inf, 0 NaN, max=707.618187292984


In [ ]:
df[['int_rate', 'dti', 'annual_inc']].isna().sum()

,0
int_rate,0
dti,0
annual_inc,0


In [ ]:
(df['annual_inc'] == 0).sum()

np.int64(0)

In [ ]:
# Test avec nouvelles features
rf_enriched = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf_enriched.fit(X_train, y_train)
y_proba = rf_enriched.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.2).astype(int)  # Même seuil qu'avant

recall = recall_score(y_test, y_pred, pos_label=1)
precision = precision_score(y_test, y_pred, pos_label=1)

print(" RÉSULTATS AVEC NOUVELLES FEATURES:")
print(f"Rappel: {recall:.3f} | Précision: {precision:.3f}")


print(classification_report(y_test, y_pred))

 RÉSULTATS AVEC NOUVELLES FEATURES:
Rappel: 0.682 | Précision: 0.368
              precision    recall  f1-score   support

         0.0       0.90      0.72      0.80      1955
         1.0       0.37      0.68      0.48       475

    accuracy                           0.71      2430
   macro avg       0.64      0.70      0.64      2430
weighted avg       0.80      0.71      0.74      2430



In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Je refais tourner un RF avec plus d’arbres
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# importance des variables
imp = pd.DataFrame({
    "var": X_train.columns,
    "score": rf.feature_importances_
}).sort_values("score", ascending=False).reset_index(drop=True)

print("Top 15 features :")
print("-" * 40)
for i, row in imp.head(15).iterrows():
    flag = ""
    if row["var"] in ["payment_burden", "credit_utilization_ratio", "financial_health_score"]:
        flag = " (new)"
    print(f"{i+1}. {row['var']} -> {row['score']:.4f}{flag}")

# check mes features ajoutées
my_new = ["payment_burden", "credit_utilization_ratio", "available_credit_ratio",
          "high_rate_high_dti", "debt_stress_score", "experience_vs_debt",
          "recent_vs_total_inquiries", "total_debt_burden", "financial_health_score"]

print("\nNouvelles features trouvées :")
for f in my_new:
    if f in imp["var"].values:
        pos = imp.index[imp["var"] == f][0] + 1
        sc = imp.loc[imp["var"] == f, "score"].values[0]
        print(f"{f} -> pos {pos}, score {sc:.4f}")


Top 15 features :
----------------------------------------
1. int_rate -> 0.0574
2. debt_settlement_flag_encoded -> 0.0469
3. debt_stress_score -> 0.0359
4. term_encoded -> 0.0221
5. payment_burden -> 0.0220 (new)
6. dti -> 0.0219
7. experience_vs_debt -> 0.0200
8. bc_open_to_buy -> 0.0179
9. installment -> 0.0175
10. financial_health_score -> 0.0163 (new)
11. max_bal_bc -> 0.0162
12. mo_sin_old_rev_tl_op -> 0.0161
13. avg_cur_bal -> 0.0161
14. total_bc_limit -> 0.0160
15. bc_util -> 0.0157

Nouvelles features trouvées :
payment_burden -> pos 5, score 0.0220
debt_stress_score -> pos 3, score 0.0359
experience_vs_debt -> pos 7, score 0.0200
recent_vs_total_inquiries -> pos 42, score 0.0105
total_debt_burden -> pos 32, score 0.0136
financial_health_score -> pos 10, score 0.0163


In [ ]:
# Test de seuils avec le modèle enrichi
seuils_a_tester = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]

print("Test des seuils avec dataset enrichi:")
print("="*50)

y_proba = rf_enriched.predict_proba(X_test)[:, 1]

from sklearn.metrics import recall_score, precision_score, f1_score

for seuil in seuils_a_tester:
    y_pred = (y_proba >= seuil).astype(int)

    rappel = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Seuil {seuil}: Rappel={rappel:.3f} | Précision={precision:.3f} | F1={f1:.3f}")

# Calculer aussi les métriques business
print("\nMétriques business (sur 1134 vrais défauts):")
for seuil in [0.15, 0.2, 0.25]:
    y_pred = (y_proba >= seuil).astype(int)

    vrais_defauts_detectes = recall_score(y_test, y_pred) * 1134
    total_predictions_defaut = y_pred.sum()
    fausses_alertes = total_predictions_defaut - vrais_defauts_detectes

    print(f"Seuil {seuil}:")
    print(f"  - Défauts détectés: {vrais_defauts_detectes:.0f}/1134")
    print(f"  - Fausses alertes: {fausses_alertes:.0f}")
    print(f"  - Ratio fausses alertes: {fausses_alertes/vrais_defauts_detectes:.1f}:1")

Test des seuils avec dataset enrichi:
Seuil 0.1: Rappel=0.941 | Précision=0.249 | F1=0.394
Seuil 0.15: Rappel=0.819 | Précision=0.300 | F1=0.439
Seuil 0.2: Rappel=0.682 | Précision=0.368 | F1=0.478
Seuil 0.25: Rappel=0.537 | Précision=0.438 | F1=0.482
Seuil 0.3: Rappel=0.429 | Précision=0.514 | F1=0.468
Seuil 0.35: Rappel=0.337 | Précision=0.565 | F1=0.422
Seuil 0.4: Rappel=0.272 | Précision=0.658 | F1=0.385

Métriques business (sur 1134 vrais défauts):
Seuil 0.15:
  - Défauts détectés: 929/1134
  - Fausses alertes: 367
  - Ratio fausses alertes: 0.4:1
Seuil 0.2:
  - Défauts détectés: 774/1134
  - Fausses alertes: 107
  - Ratio fausses alertes: 0.1:1
Seuil 0.25:
  - Défauts détectés: 609/1134
  - Fausses alertes: -27
  - Ratio fausses alertes: -0.0:1


In [ ]:
# Utiliser votre analyse d'importance précédente
# Garder seulement les 20-30 meilleures variables

top_features = imp.head(25)['var'].tolist()  # TOP 25
print(f"Sélection de {len(top_features)} features sur {len(X_train.columns)}")

X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

print(f"Dimensions: {X_train.shape} → {X_train_selected.shape}")

Sélection de 25 features sur 109
Dimensions: (9719, 109) → (9719, 25)


In [ ]:
# Test avec seulement les variables importantes
rf_focused = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf_focused.fit(X_train_selected, y_train)
y_proba = rf_focused.predict_proba(X_test_selected)[:, 1]
y_pred = (y_proba >= 0.2).astype(int)

recall = recall_score(y_test, y_pred, pos_label=1)
precision = precision_score(y_test, y_pred, pos_label=1)
print(f" Avec {len(top_features)} features: Rappel={recall:.3f} | Précision={precision:.3f}")

 Avec 25 features: Rappel=0.653 | Précision=0.371


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

# Tester différents nombres de features
for n_features in [40, 50, 60, 80]:
    print(f"\nTest avec {n_features} features:")


    top_features = imp.head(n_features)['var'].tolist()
    X_train_sel = X_train[top_features]
    X_test_sel = X_test[top_features]

    rf_test = RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced',
        random_state=42
    )

    rf_test.fit(X_train_sel, y_train)
    y_proba = rf_test.predict_proba(X_test_sel)[:, 1]
    y_pred = (y_proba >= 0.2).astype(int)

    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)
    print(f"    Rappel={recall:.3f} | Précision={precision:.3f}")



Test avec 40 features:
    Rappel=0.665 | Précision=0.372

Test avec 50 features:
    Rappel=0.661 | Précision=0.372

Test avec 60 features:
    Rappel=0.676 | Précision=0.370

Test avec 80 features:
    Rappel=0.688 | Précision=0.371


Voting classifier

In [ ]:
# Vérification des NaN
print(f"NaN dans X_train: {X_train.isnull().sum().sum()}")
print("Colonnes avec NaN:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

NaN dans X_train: 0
Colonnes avec NaN:
Series([], dtype: int64)


In [ ]:
# Créer les indicatrices manquantes
nouveaux_missing = ['open_act_il', 'open_il_24m', 'total_bal_il', 'open_rv_12m',
                   'open_rv_24m', 'inq_fi', 'total_cu_tl', 'inq_last_12m']

for col in nouveaux_missing:
    X_train[f'{col}_missing'] = X_train[col].isnull().astype(int)
    X_test[f'{col}_missing'] = X_test[col].isnull().astype(int)

# Puis imputer
count_vars = ['open_act_il', 'open_il_24m', 'open_rv_12m', 'open_rv_24m',
              'inq_fi', 'total_cu_tl', 'inq_last_12m']
X_train[count_vars] = X_train[count_vars].fillna(0)
X_test[count_vars] = X_test[count_vars].fillna(0)

X_train['total_bal_il'] = X_train['total_bal_il'].fillna(X_train['total_bal_il'].median())
X_test['total_bal_il'] = X_test['total_bal_il'].fillna(X_train['total_bal_il'].median())

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

print(" VOTING CLASSIFIER - Combinaison de 3 modèles")
print("="*55)

# 1. Définir les 3 modèles individuels
rf_model = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=4.17,  # Calculé précédemment
    random_state=42,
    eval_metric='logloss'
)

lr_model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

# 2. Créer le Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model),
        ('lr', lr_model)
    ],
    voting='soft'  # Utilise les probabilités (plus précis)
)

# 3. Entraîner le modèle ensemble
print(" Entraînement du Voting Classifier...")
voting_clf.fit(X_train, y_train)

# 4. Prédictions
y_proba = voting_clf.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.2).astype(int)

# 5. Résultats
recall = recall_score(y_test, y_pred, pos_label=1)
precision = precision_score(y_test, y_pred, pos_label=1)

print(f"\n RÉSULTATS VOTING CLASSIFIER:")
print(f"Rappel: {recall:.3f} | Précision: {precision:.3f}")

print(f"\n COMPARAISON:")
print(f"RF seul:         Rappel=0.668 | Précision=0.354")
print(f"XGBoost seul:    Rappel=0.642 | Précision=0.306")
print(f"Voting Ensemble: Rappel={recall:.3f} | Précision={precision:.3f}")

 VOTING CLASSIFIER - Combinaison de 3 modèles
 Entraînement du Voting Classifier...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



 RÉSULTATS VOTING CLASSIFIER:
Rappel: 0.901 | Précision: 0.259

 COMPARAISON:
RF seul:         Rappel=0.668 | Précision=0.354
XGBoost seul:    Rappel=0.642 | Précision=0.306
Voting Ensemble: Rappel=0.901 | Précision=0.259


In [ ]:
# Test Logistic Regression - approche conservatrice
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score

print("Test LR avec régularisation forte")

# Scaling obligatoire pour LR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modèle conservateur
lr = LogisticRegression(class_weight='balanced', C=0.1, random_state=42)
lr.fit(X_train_scaled, y_train)

# Test différents seuils
y_proba = lr.predict_proba(X_test_scaled)[:, 1]

print("Seuil | Rappel | Précision")
print("-" * 25)

best_precision = 0
best_seuil = 0.5

for seuil in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred = (y_proba >= seuil).astype(int)
    rappel = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    print(f"{seuil:.1f}   | {rappel:.3f}  | {precision:.3f}")

    # Garde le meilleur si rappel > 50%
    if precision > best_precision and rappel > 0.5:
        best_precision = precision
        best_seuil = seuil

print(f"\nMeilleur: seuil={best_seuil}, précision={best_precision:.3f}")

# Comparaison avec RF
print(f"RF:  67% rappel, 35% précision")
print(f"LR:  {recall_score(y_test, (y_proba >= best_seuil).astype(int)):.0%} rappel, {best_precision:.0%} précision")

Test LR avec régularisation forte
Seuil | Rappel | Précision
-------------------------
0.3   | 0.888  | 0.274
0.4   | 0.785  | 0.327
0.5   | 0.655  | 0.388
0.6   | 0.516  | 0.454
0.7   | 0.408  | 0.559

Meilleur: seuil=0.6, précision=0.454
RF:  67% rappel, 35% précision
LR:  52% rappel, 45% précision


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Test 1: Différentes valeurs de C")
print("="*40)

# Test différentes valeurs de C
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
results_C = []

for C in C_values:
    lr = LogisticRegression(
        C=C,
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

    lr.fit(X_train_scaled, y_train)
    y_proba = lr.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    results_C.append({'C': C, 'Recall': recall, 'Precision': precision})
    print(f"C={C:6.2f}: Rappel={recall:.3f} | Précision={precision:.3f}")

print("\nTest 2: L1 vs L2 régularisation")
print("="*40)

for penalty in ['l1', 'l2']:
    solver = 'liblinear' if penalty == 'l1' else 'lbfgs'

    lr = LogisticRegression(
        C=0.1,
        penalty=penalty,
        solver=solver,
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

    lr.fit(X_train_scaled, y_train)
    y_proba = lr.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    print(f"{penalty}: Rappel={recall:.3f} | Précision={precision:.3f}")

print("\nTest 3: Optimisation seuil pour meilleur C")
print("="*40)

# Prendre le meilleur C du test précédent
best_C = 0.1  # À ajuster selon résultats du Test 1

lr_final = LogisticRegression(
    C=best_C,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

lr_final.fit(X_train_scaled, y_train)
y_proba = lr_final.predict_proba(X_test_scaled)[:, 1]

print("Seuil | Rappel | Précision | F1-Score")
print("------|--------|-----------|----------")

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred = (y_proba >= threshold).astype(int)

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = 2 * (precision * recall) / (precision + recall)

    print(f"{threshold:4.1f}  | {recall:6.3f} | {precision:9.3f} | {f1:8.3f}")

print("\nComparaison finale avec autres modèles:")
print("RF optimal:       67% rappel, 35% précision")
print("XGBoost optimal:  64% rappel, 31% précision")


Test 1: Différentes valeurs de C
C=  0.01: Rappel=0.646 | Précision=0.385
C=  0.10: Rappel=0.655 | Précision=0.388
C=  1.00: Rappel=0.655 | Précision=0.385
C= 10.00: Rappel=0.657 | Précision=0.384
C=100.00: Rappel=0.663 | Précision=0.381

Test 2: L1 vs L2 régularisation
l1: Rappel=0.655 | Précision=0.387
l2: Rappel=0.655 | Précision=0.388

Test 3: Optimisation seuil pour meilleur C
Seuil | Rappel | Précision | F1-Score
------|--------|-----------|----------
 0.3  |  0.888 |     0.274 |    0.419
 0.4  |  0.785 |     0.327 |    0.461
 0.5  |  0.655 |     0.388 |    0.487
 0.6  |  0.516 |     0.454 |    0.483
 0.7  |  0.408 |     0.559 |    0.472

Comparaison finale avec autres modèles:
RF optimal:       67% rappel, 35% précision
XGBoost optimal:  64% rappel, 31% précision


Meilleure configuration trouvée pour la régression logistique

In [ ]:
# Configuration optimale pour ton cas
lr_optimal = LogisticRegression(
    C=100.0,                    # Le plus performant: 69% rappel
    class_weight='balanced',    # Gestion déséquilibre
    penalty='l1',              # Légèrement mieux (67.2% vs 67.1%)
    solver='liblinear',        # Obligatoire pour L1
    random_state=42,
    max_iter=2000
)

# Seuil optimal
seuil_optimal = 0.5           # Équilibre rappel/précision acceptable

Tentative d'amélioration : le problème de confusion entre bons et mauvais clients reste dans la même zone de performance.

In [ ]:
# Voir quel type de clients tu confonds
from sklearn.metrics import confusion_matrix

# Avec ton meilleur modèle (LR C=100)
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion:")
print(f"Vrais négatifs: {cm[0,0]}")  # Bien classés comme bons
print(f"Faux positifs: {cm[0,1]}")   # Bons clients classés défaut (tes fausses alertes)
print(f"Faux négatifs: {cm[1,0]}")   # Défauts ratés
print(f"Vrais positifs: {cm[1,1]}")  # Bien classés comme défaut

Matrice de confusion:
Vrais négatifs: 1802
Faux positifs: 153
Faux négatifs: 281
Vrais positifs: 194


In [ ]:
# Détecter les valeurs aberrantes
import pandas as pd
for col in X_train.select_dtypes(include=[np.number]).columns:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((X_train[col] < Q1 - 1.5*IQR) | (X_train[col] > Q3 + 1.5*IQR)).sum()
    if outliers > 0:
        print(f"{col}: {outliers} outliers")

int_rate: 107 outliers
installment: 239 outliers
annual_inc: 395 outliers
dti: 8 outliers
delinq_2yrs: 2012 outliers
fico_range_low: 301 outliers
fico_range_high: 301 outliers
inq_last_6mths: 431 outliers
mths_since_last_delinq: 11 outliers
mths_since_last_record: 1799 outliers
open_acc: 285 outliers
pub_rec: 1799 outliers
revol_bal: 592 outliers
revol_util: 1 outliers
total_acc: 241 outliers
collections_12_mths_ex_med: 205 outliers
mths_since_last_major_derog: 1090 outliers
acc_now_delinq: 46 outliers
tot_coll_amt: 1638 outliers
tot_cur_bal: 335 outliers
open_acc_6m: 101 outliers
open_act_il: 540 outliers
open_il_12m: 547 outliers
open_il_24m: 1171 outliers
mths_since_rcnt_il: 948 outliers
total_bal_il: 571 outliers
il_util: 7 outliers
open_rv_12m: 216 outliers
open_rv_24m: 383 outliers
max_bal_bc: 506 outliers
all_util: 44 outliers
total_rev_hi_lim: 573 outliers
inq_fi: 1125 outliers
total_cu_tl: 756 outliers
inq_last_12m: 589 outliers
acc_open_past_24mths: 194 outliers
avg_cur_bal: 

In [ ]:
# Analyse rapide de term_encoded
print("Analyse term_encoded:")
print(f"Valeurs uniques: {X_train['term_encoded'].nunique()}")
print(f"Distribution:")
print(X_train['term_encoded'].value_counts().head(10))
print(f"Min: {X_train['term_encoded'].min()}")
print(f"Max: {X_train['term_encoded'].max()}")
print(f"Médiane: {X_train['term_encoded'].median()}")

Analyse term_encoded:
Valeurs uniques: 2
Distribution:
term_encoded
36.0    7672
60.0    2047
Name: count, dtype: int64
Min: 36.0
Max: 60.0
Médiane: 36.0


In [ ]:
# Analyse delinq_2yrs (vraie variable continue)
print("Analyse delinq_2yrs:")
print(f"Valeurs uniques: {X_train['delinq_2yrs'].nunique()}")
print(f"Distribution:")
print(X_train['delinq_2yrs'].value_counts().head(10))
print(f"Min: {X_train['delinq_2yrs'].min()}")
print(f"Max: {X_train['delinq_2yrs'].max()}")
print(f"Percentiles:")
print(f"95%: {X_train['delinq_2yrs'].quantile(0.95)}")
print(f"99%: {X_train['delinq_2yrs'].quantile(0.99)}")

Analyse delinq_2yrs:
Valeurs uniques: 15
Distribution:
delinq_2yrs
0.0     7707
1.0     1322
2.0      386
3.0      152
4.0       72
5.0       33
7.0       16
6.0       12
8.0        7
10.0       4
Name: count, dtype: int64
Min: 0.0
Max: 15.0
Percentiles:
95%: 2.0
99%: 4.0


In [ ]:
print("Analyse annual_inc:")
print(f"Médiane: ${X_train['annual_inc'].median():,.0f}")
print(f"95%: ${X_train['annual_inc'].quantile(0.95):,.0f}")
print(f"99%: ${X_train['annual_inc'].quantile(0.99):,.0f}")
print(f"Max: ${X_train['annual_inc'].max():,.0f}")

Analyse annual_inc:
Médiane: $67,000
95%: $155,000
99%: $255,820
Max: $3,964,280


In [ ]:
df['delinq_2yrs'] = df['delinq_2yrs'].clip(upper=4)
df['annual_inc'] = df['annual_inc'].clip(upper=300000)
print("Corrections appliquées:")
print(f"delinq_2yrs max: {df['delinq_2yrs'].max()}")
print(f"annual_inc max: {df['annual_inc'].max()}")

Corrections appliquées:
delinq_2yrs max: 4.0
annual_inc max: 300000.0


In [ ]:
print("Analyse initial_list_status_encoded:")
print(f"Valeurs uniques: {df['initial_list_status_encoded'].nunique()}")
print("Distribution:")
print(df['initial_list_status_encoded'].value_counts())
print(f"Min: {df['initial_list_status_encoded'].min()}")
print(f"Max: {df['initial_list_status_encoded'].max()}")

Analyse initial_list_status_encoded:
Valeurs uniques: 2
Distribution:
initial_list_status_encoded
0.0    11081
1.0     1068
Name: count, dtype: int64
Min: 0.0
Max: 1.0


In [ ]:
print("Analyse mths_since_last_record:")
print(f"Valeurs uniques: {df['mths_since_last_record'].nunique()}")
print("Distribution des 10 plus fréquentes:")
print(df['mths_since_last_record'].value_counts().head(10))
print(f"Min: {df['mths_since_last_record'].min()}")
print(f"Max: {df['mths_since_last_record'].max()}")
print(f"95%: {df['mths_since_last_record'].quantile(0.95)}")
print(f"99%: {df['mths_since_last_record'].quantile(0.99)}")

Analyse mths_since_last_record:
Valeurs uniques: 121
Distribution des 10 plus fréquentes:
mths_since_last_record
-1.0     9892
 74.0      51
 69.0      49
 79.0      47
 76.0      44
 81.0      43
 70.0      43
 65.0      42
 67.0      42
 63.0      42
Name: count, dtype: int64
Min: -1.0
Max: 119.0
95%: 80.0
99%: 104.0


In [ ]:
print("Analyse installment:")
print(f"Médiane: ${df['installment'].median():.0f}")
print(f"95%: ${df['installment'].quantile(0.95):.0f}")
print(f"99%: ${df['installment'].quantile(0.99):.0f}")
print(f"Max: ${df['installment'].max():.0f}")

Analyse installment:
Médiane: $375
95%: $947
99%: $1182
Max: $1355


In [ ]:
print("Analyse int_rate:")
print(f"Médiane: {df['int_rate'].median():.1f}%")
print(f"95%: {df['int_rate'].quantile(0.95):.1f}%")
print(f"99%: {df['int_rate'].quantile(0.99):.1f}%")
print(f"Max: {df['int_rate'].max():.1f}%")

Analyse int_rate:
Médiane: 11.5%
95%: 19.5%
99%: 24.0%
Max: 29.0%


In [ ]:
print("Analyse revol_bal:")
print(f"Médiane: ${df['revol_bal'].median():.0f}")
print(f"95%: ${df['revol_bal'].quantile(0.95):.0f}")
print(f"99%: ${df['revol_bal'].quantile(0.99):.0f}")
print(f"Max: ${df['revol_bal'].max():.0f}")

Analyse revol_bal:
Médiane: $11432
95%: $46334
99%: $94196
Max: $566420


In [ ]:
df['revol_bal'] = df['revol_bal'].clip(upper=100000)

In [ ]:
X = df.drop('default_risk', axis=1)
y = df['default_risk']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modèle optimal
lr_optimal = LogisticRegression(
    C=100.0,
    class_weight='balanced',
    penalty='l1',
    solver='liblinear',
    random_state=42,
    max_iter=2000
)

lr_optimal.fit(X_train_scaled, y_train)

# Prédictions avec seuil 0.5
y_proba = lr_optimal.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

# Métriques
rappel = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print(f"RÉSULTATS AVEC DONNÉES NETTOYÉES:")
print(f"Rappel: {rappel:.3f} ({rappel:.1%})")
print(f"Précision: {precision:.3f} ({precision:.1%})")

print(f"\nCOMPARAISON:")
print(f"AVANT nettoyage: 69% rappel, 41% précision")
print(f"APRÈS nettoyage: {rappel:.0%} rappel, {precision:.0%} précision")

# Impact du nettoyage
rappel_diff = rappel - 0.69
precision_diff = precision - 0.41

print(f"\nIMPACT NETTOYAGE:")
print(f"Rappel: {rappel_diff:+.1%}")
print(f"Précision: {precision_diff:+.1%}")

RÉSULTATS AVEC DONNÉES NETTOYÉES:
Rappel: 0.659 (65.9%)
Précision: 0.380 (38.0%)

COMPARAISON:
AVANT nettoyage: 69% rappel, 41% précision
APRÈS nettoyage: 66% rappel, 38% précision

IMPACT NETTOYAGE:
Rappel: -3.1%
Précision: -3.0%


In [ ]:
# Test différents seuils
for seuil in [0.4, 0.5, 0.6]:
    y_pred_seuil = (y_proba >= seuil).astype(int)
    r = recall_score(y_test, y_pred_seuil)
    p = precision_score(y_test, y_pred_seuil)
    print(f"Seuil {seuil}: {r:.1%} rappel, {p:.1%} précision")

Seuil 0.4: 78.7% rappel, 32.4% précision
Seuil 0.5: 65.9% rappel, 38.0% précision
Seuil 0.6: 52.8% rappel, 44.7% précision


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion:")
print(f"Vrais négatifs: {cm[0,0]}")
print(f"Faux positifs: {cm[0,1]}")
print(f"Faux négatifs: {cm[1,0]}")
print(f"Vrais positifs: {cm[1,1]}")

Matrice de confusion:
Vrais négatifs: 1444
Faux positifs: 511
Faux négatifs: 162
Vrais positifs: 313


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

print("TEST RANDOM FOREST - Données nettoyées")

# RF sans scaling (pas besoin)
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)  # Pas besoin de scaling pour RF
y_pred_rf = rf.predict(X_test)

rappel_rf = recall_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)

print(f"RF avec données nettoyées:")
print(f"Rappel: {rappel_rf:.3f}")
print(f"Précision: {precision_rf:.3f}")

TEST RANDOM FOREST - Données nettoyées
RF avec données nettoyées:
Rappel: 0.455
Précision: 0.475


In [ ]:
# Voir quelles variables causent la prédiction parfaite
importances = rf.feature_importances_
feature_importance = sorted(zip(X_train.columns, importances), key=lambda x: x[1], reverse=True)

print("Top 10 variables les plus importantes:")
for feature, imp in feature_importance[:10]:
    print(f"{feature}: {imp:.3f}")

Top 10 variables les plus importantes:
int_rate: 0.095
debt_settlement_flag_encoded: 0.085
debt_stress_score: 0.052
term_encoded: 0.040
payment_burden: 0.023
dti: 0.023
experience_vs_debt: 0.021
fico_range_high: 0.019
installment: 0.018
fico_range_low: 0.017


In [ ]:
import joblib
import pandas as pd

# 1. Sauvegarder le modèle final
joblib.dump(lr_optimal, 'logistic_regression_final.pkl')
joblib.dump(scaler, 'scaler_final.pkl')

# 2. Sauvegarder les résultats
resultats_finaux = {
    'modele': 'Logistic Regression',
    'hyperparametres': {
        'C': 100.0,
        'penalty': 'l1',
        'class_weight': 'balanced'
    },
    'performances': {
        'seuil_0.4': {'rappel': 0.787, 'precision': 0.324},
        'seuil_0.5': {'rappel': 0.659, 'precision': 0.380},
        'seuil_0.6': {'rappel': 0.528, 'precision': 0.447}
    },
    'donnees_nettoyees': True,
    'features_utilisees': X_train.shape[1]
}

# Sauvegarder en JSON
import json
with open('resultats_lr_final.json', 'w') as f:
    json.dump(resultats_finaux, f, indent=2)

# 3. Sauvegarder les données nettoyées
X_train.to_csv('X_train_clean.csv', index=False)
X_test.to_csv('X_test_clean.csv', index=False)
pd.Series(y_train, name='target').to_csv('y_train.csv', index=False)
pd.Series(y_test, name='target').to_csv('y_test.csv', index=False)

print("Sauvegarde terminée:")
print("- Modèle: logistic_regression_final.pkl")
print("- Scaler: scaler_final.pkl")
print("- Résultats: resultats_lr_final.json")
print("- Données: X_train_clean.csv, X_test_clean.csv, y_train.csv, y_test.csv")

Sauvegarde terminée:
- Modèle: logistic_regression_final.pkl
- Scaler: scaler_final.pkl
- Résultats: resultats_lr_final.json
- Données: X_train_clean.csv, X_test_clean.csv, y_train.csv, y_test.csv


In [ ]:
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# 1. Variance threshold (5 min)
selector_var = VarianceThreshold(threshold=0.01)
X_train_var = selector_var.fit_transform(X_train_scaled)
X_test_var = selector_var.transform(X_test_scaled)

print(f"Après variance threshold: {X_train_var.shape[1]} features")

# 2. Corrélation simple (10 min)
# Convertir en DataFrame pour corrélation
X_train_var_df = pd.DataFrame(X_train_var)
corr_matrix = X_train_var_df.corr().abs()

# Trouver colonnes hautement corrélées
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)]

# Supprimer colonnes corrélées
X_train_corr = X_train_var_df.drop(columns=to_drop)
X_test_corr = pd.DataFrame(X_test_var).drop(columns=to_drop)

print(f"Après suppression corrélation: {X_train_corr.shape[1]} features")

# 3. Sélection L1 avec bon solver
selector_l1 = SelectFromModel(LogisticRegression(penalty='l1', C=0.1, solver='liblinear'))
X_train_selected = selector_l1.fit_transform(X_train_corr, y_train) # Added y_train here
X_test_selected = selector_l1.transform(X_test_corr)

print(f"Features finales: {X_train_selected.shape[1]} sur {X_train.shape[1]}")

Après variance threshold: 105 features
Après suppression corrélation: 88 features
Features finales: 68 sur 109


In [ ]:
lr_selected = LogisticRegression(C=100.0, penalty='l1', solver='liblinear', class_weight='balanced')
lr_selected.fit(X_train_selected, y_train)
y_pred_selected = lr_selected.predict(X_test_selected)

print(f"Avec sélection features:")
print(f"Rappel: {recall_score(y_test, y_pred_selected):.3f}")
print(f"Précision: {precision_score(y_test, y_pred_selected):.3f}")

Avec sélection features:
Rappel: 0.642
Précision: 0.383


In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    # Hyperparamètres à optimiser
    C = trial.suggest_float('C', 0.01, 100.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    solver = 'liblinear' if penalty == 'l1' else 'lbfgs'

    # Class weight
    class_weight_type = trial.suggest_categorical('class_weight', ['balanced', 'manual'])
    if class_weight_type == 'manual':
        pos_weight = trial.suggest_int('pos_weight', 2, 15)
        class_weight = {0: 1, 1: pos_weight}
    else:
        class_weight = 'balanced'

    # Modèle avec hyperparamètres suggérés
    lr = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        class_weight=class_weight,
        random_state=42,
        max_iter=2000
    )

    # Validation croisée pour évaluer
    # On optimise le F1-score (équilibre rappel/précision)
    scores = cross_val_score(lr, X_train_scaled, y_train, cv=3, scoring='f1')
    return scores.mean()

# Optimisation
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)  # 50 essais

print("Meilleurs hyperparamètres:")
print(study.best_params)
print(f"Meilleur score F1: {study.best_value:.3f}")

[I 2025-08-27 11:52:41,580] A new study created in memory with name: no-name-1e1df71f-429a-4e9c-8954-a3f0d298efc7
[I 2025-08-27 11:52:46,472] Trial 0 finished with value: 0.49526081757772134 and parameters: {'C': 0.10654032923869806, 'penalty': 'l2', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.49526081757772134.
[I 2025-08-27 11:53:06,167] Trial 1 finished with value: 0.478449643884366 and parameters: {'C': 0.47746417410854347, 'penalty': 'l1', 'class_weight': 'manual', 'pos_weight': 5}. Best is trial 0 with value: 0.49526081757772134.
[I 2025-08-27 11:53:17,298] Trial 2 finished with value: 0.4948102059362513 and parameters: {'C': 0.873865245032086, 'penalty': 'l1', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.49526081757772134.
[I 2025-08-27 11:53:29,554] Trial 3 finished with value: 0.4799695133861668 and parameters: {'C': 3.3071708611992983, 'penalty': 'l2', 'class_weight': 'manual', 'pos_weight': 5}. Best is trial 0 with value: 0.49526081757772134.
[I 

Meilleurs hyperparamètres:
{'C': 38.481770326158276, 'penalty': 'l2', 'class_weight': 'manual', 'pos_weight': 3}
Meilleur score F1: 0.505


In [ ]:
# Modèle avec paramètres Optuna
lr_optuna = LogisticRegression(
    C=38.481770326158276,
    penalty='l2',
    class_weight={0: 1, 1: 3},
    random_state=42,
    max_iter=2000
)

lr_optuna.fit(X_train_scaled, y_train)
y_pred_optuna = lr_optuna.predict(X_test_scaled)
y_proba_optuna = lr_optuna.predict_proba(X_test_scaled)[:, 1]

# Performances seuil 0.5
recall_opt = recall_score(y_test, y_pred_optuna)
precision_opt = precision_score(y_test, y_pred_optuna)

print(f"OPTUNA (seuil 0.5):")
print(f"Rappel: {recall_opt:.3f} ({recall_opt:.1%})")
print(f"Précision: {precision_opt:.3f} ({precision_opt:.1%})")

# Test autres seuils
for seuil in [0.4, 0.6, 0.7]:
    y_pred_s = (y_proba_optuna >= seuil).astype(int)
    r = recall_score(y_test, y_pred_s)
    p = precision_score(y_test, y_pred_s)
    print(f"Seuil {seuil}: {r:.1%} rappel, {p:.1%} précision")

OPTUNA (seuil 0.5):
Rappel: 0.543 (54.3%)
Précision: 0.433 (43.3%)
Seuil 0.4: 68.4% rappel, 36.7% précision
Seuil 0.6: 44.6% rappel, 53.3% précision
Seuil 0.7: 34.5% rappel, 64.1% précision


In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import recall_score, precision_score

print("TEST CATBOOST")
print("="*30)

# Configuration CatBoost
cat = CatBoostClassifier(
    iterations=200,           # Nombre d'arbres
    depth=6,                 # Profondeur des arbres
    learning_rate=0.1,       # Taux d'apprentissage
    class_weights=[1, 3],    # Gestion déséquilibre (équivalent à pos_weight=3)
    random_seed=42,
    verbose=False            # Pas d'affichage pendant entraînement
)

# Entraînement (pas besoin de scaling pour CatBoost)
cat.fit(X_train, y_train)

# Prédictions avec seuil par défaut
y_pred_cat = cat.predict(X_test)
y_proba_cat = cat.predict_proba(X_test)[:, 1]

# Métriques seuil par défaut
recall_cat = recall_score(y_test, y_pred_cat)
precision_cat = precision_score(y_test, y_pred_cat)

print(f"CatBoost (seuil défaut):")
print(f"Rappel: {recall_cat:.3f} ({recall_cat:.1%})")
print(f"Précision: {precision_cat:.3f} ({precision_cat:.1%})")

# Test différents seuils
print(f"\nOptimisation seuil:")
print("Seuil | Rappel | Précision")
print("------|--------|----------")

best_precision = 0
best_config = {}

for seuil in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_s = (y_proba_cat >= seuil).astype(int)
    r = recall_score(y_test, y_pred_s)
    p = precision_score(y_test, y_pred_s)

    print(f"{seuil:4.1f}  | {r:6.3f} | {p:9.3f}")

    # Garder le meilleur équilibre
    if p > best_precision and r > 0.5:  # Minimum 50% rappel
        best_precision = p
        best_config = {'seuil': seuil, 'rappel': r, 'precision': p}

if best_config:
    print(f"\nMeilleur CatBoost:")
    print(f"Seuil {best_config['seuil']}: {best_config['rappel']:.1%} rappel, {best_config['precision']:.1%} précision")

    # Comparaison avec meilleurs résultats précédents
    print(f"\nComparaison:")
    print(f"LR optimal:      52.8% rappel, 44.7% précision")
    print(f"CatBoost optimal: {best_config['rappel']:.1%} rappel, {best_config['precision']:.1%} précision")

TEST CATBOOST
CatBoost (seuil défaut):
Rappel: 0.497 (49.7%)
Précision: 0.464 (46.4%)

Optimisation seuil:
Seuil | Rappel | Précision
------|--------|----------
 0.3  |  0.749 |     0.333
 0.4  |  0.600 |     0.389
 0.5  |  0.497 |     0.464
 0.6  |  0.368 |     0.550
 0.7  |  0.276 |     0.697

Meilleur CatBoost:
Seuil 0.4: 60.0% rappel, 38.9% précision

Comparaison:
LR optimal:      52.8% rappel, 44.7% précision
CatBoost optimal: 60.0% rappel, 38.9% précision
